# Feature Store Streaming Demo - Setup

Run once to set up the environment:
1. Database, schema, warehouse, tables
2. Seed USERS, ITEMS, RAW_EVENTS
3. Register entities + online service
4. Register feature views (batch + stream)

In [ ]:
!pip install --upgrade "snowflake-ml-python>=1.41" snowflake-connector-python requests

In [ ]:
%%sql -r dataframe_1
CREATE WAREHOUSE IF NOT EXISTS ML_DEMO_WH WAREHOUSE_SIZE='XSMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
CREATE DATABASE IF NOT EXISTS ML_DEMOS;
CREATE SCHEMA IF NOT EXISTS ML_DEMOS.ONLINE_W_STREAMING;
USE DATABASE ML_DEMOS;
USE SCHEMA ONLINE_W_STREAMING;
USE WAREHOUSE ML_DEMO_WH;

CREATE TABLE IF NOT EXISTS USERS (
    user_id VARCHAR(20) NOT NULL, signup_ts TIMESTAMP_NTZ,
    country VARCHAR(50), user_segment VARCHAR(20),
    device_type VARCHAR(20), age_group VARCHAR(20),
    PRIMARY KEY (user_id));

CREATE TABLE IF NOT EXISTS ITEMS (
    item_id VARCHAR(20) NOT NULL, category VARCHAR(50) NOT NULL,
    subcategory VARCHAR(50), brand VARCHAR(50), price FLOAT,
    avg_rating FLOAT, is_available BOOLEAN DEFAULT TRUE,
    created_at TIMESTAMP_NTZ, PRIMARY KEY (item_id));

CREATE TABLE IF NOT EXISTS RAW_EVENTS (
    event_id VARCHAR(36) NOT NULL, session_id VARCHAR(36) NOT NULL,
    user_id VARCHAR(20) NOT NULL, item_id VARCHAR(20),
    event_type VARCHAR(20) NOT NULL, event_ts TIMESTAMP_NTZ NOT NULL,
    dwell_time_sec INT, search_query VARCHAR(200),
    referrer_item_id VARCHAR(20), properties VARIANT,
    PRIMARY KEY (event_id));

In [ ]:
import random
from datetime import datetime, timedelta
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

session = get_active_session()
session.use_schema("ML_DEMOS.ONLINE_W_STREAMING")
session.use_warehouse("ML_DEMO_WH")

random.seed(42)

COUNTRIES = ["US", "CA", "UK", "DE", "FR", "JP", "AU", "BR", "IN", "MX"]
SEGMENTS = ["new", "active", "power", "dormant"]
DEVICES = ["mobile", "desktop", "tablet"]
AGE_GROUPS = ["18-24", "25-34", "35-44", "45-54", "55+"]

users = []
for i in range(200):
    signup = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 800))
    users.append({
        "USER_ID": f"U{i:04d}",
        "SIGNUP_TS": signup.strftime("%Y-%m-%d %H:%M:%S"),
        "COUNTRY": random.choice(COUNTRIES),
        "USER_SEGMENT": random.choice(SEGMENTS),
        "DEVICE_TYPE": random.choice(DEVICES),
        "AGE_GROUP": random.choice(AGE_GROUPS),
    })

CATEGORIES = {
    "Running": (["Road Shoes", "Trail Shoes", "Socks", "Shorts"], ["Nike", "Adidas", "Brooks", "Hoka"]),
    "Cycling": (["Road Bikes", "Helmets", "Jerseys", "Lights"], ["Trek", "Specialized", "Giant", "Cannondale"]),
    "Yoga": (["Mats", "Blocks", "Straps", "Leggings"], ["Lululemon", "Manduka", "Gaiam", "Alo"]),
    "Swimming": (["Goggles", "Caps", "Fins", "Suits"], ["Speedo", "TYR", "Arena", "Finis"]),
    "Camping": (["Tents", "Sleeping Bags", "Stoves", "Packs"], ["REI", "MSR", "Osprey", "Big Agnes"]),
}

items = []
item_idx = 0
for category, (subcats, brands) in CATEGORIES.items():
    for sub in subcats:
        for brand in brands:
            items.append({
                "ITEM_ID": f"I{item_idx:04d}",
                "CATEGORY": category,
                "SUBCATEGORY": sub,
                "BRAND": brand,
                "PRICE": round(random.uniform(15.0, 500.0), 2),
                "AVG_RATING": round(random.uniform(3.0, 5.0), 1),
                "IS_AVAILABLE": random.random() > 0.1,
                "CREATED_AT": (datetime(2022, 6, 1) + timedelta(days=random.randint(0, 600))).strftime("%Y-%m-%d %H:%M:%S"),
            })
            item_idx += 1

users_df = session.create_dataframe(users)
users_df.write.mode("overwrite").save_as_table("ML_DEMOS.ONLINE_W_STREAMING.USERS")

items_df = session.create_dataframe(items)
items_df.write.mode("overwrite").save_as_table("ML_DEMOS.ONLINE_W_STREAMING.ITEMS")

print(f"Seeded {len(users)} users and {len(items)} items.")

In [ ]:
# Generate ~10k historical events for training (self-contained)
import uuid, json, math, time
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass, field
from typing import Optional

SESSION_TRANSITIONS = {
    "start": {"view": 0.70, "click": 0.20, "search": 0.10},
    "view": {"view": 0.55, "click": 0.20, "add_to_cart": 0.12, "search": 0.08, "end": 0.05},
    "click": {"view": 0.60, "add_to_cart": 0.20, "search": 0.10, "end": 0.10},
    "search": {"view": 0.65, "click": 0.25, "end": 0.10},
    "add_to_cart": {"view": 0.40, "purchase": 0.30, "add_to_cart": 0.15, "end": 0.15},
    "purchase": {"view": 0.30, "end": 0.70},
}
SEARCH_QUERIES = ["running shoes", "trail shoes", "road bike", "yoga mat", "protein powder",
    "basketball shoes", "camping tent", "swim goggles", "foam roller", "energy gels"]

@dataclass
class EventSession:
    session_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    user_id: str = ""
    state: str = "start"
    last_item_id: Optional[str] = None
    events_count: int = 0
    max_events: int = field(default_factory=lambda: random.randint(3, 25))
    category_bias: Optional[str] = None
    def is_done(self): return self.state == "end" or self.events_count >= self.max_events

def _weighted_choice(options):
    return random.choices(list(options.keys()), weights=list(options.values()), k=1)[0]

def gen_event(ev_session, users, items_by_cat, all_items):
    if ev_session.is_done(): return None
    user = next(u for u in users if u["USER_ID"] == ev_session.user_id)
    next_state = _weighted_choice(SESSION_TRANSITIONS.get(ev_session.state, {"end": 1.0}))
    ev_session.state = next_state
    ev_session.events_count += 1
    if next_state == "end": return None
    event = {"EVENT_ID": str(uuid.uuid4()), "SESSION_ID": ev_session.session_id,
             "USER_ID": ev_session.user_id, "ITEM_ID": None, "EVENT_TYPE": next_state,
             "EVENT_TS": None, "DWELL_TIME_SEC": None, "SEARCH_QUERY": None,
             "REFERRER_ITEM_ID": None, "PROPERTIES": json.dumps({"device": user["DEVICE_TYPE"], "country": user["COUNTRY"]})}
    if next_state == "search":
        event["SEARCH_QUERY"] = random.choice(SEARCH_QUERIES)
    elif next_state in ("view", "click", "add_to_cart", "purchase"):
        pool = items_by_cat.get(ev_session.category_bias, all_items) if ev_session.category_bias else all_items
        item_id = random.choice(pool)
        event["ITEM_ID"] = item_id
        ev_session.last_item_id = item_id
        for cat, cat_items in items_by_cat.items():
            if item_id in cat_items: ev_session.category_bias = cat; break
        if next_state == "view":
            event["DWELL_TIME_SEC"] = max(2, min(300, int(math.exp(random.gauss(2.7, 0.9)))))
    return event

user_ids = [u["USER_ID"] for u in users]
items_by_cat = {}
for item in items:
    items_by_cat.setdefault(item["CATEGORY"], []).append(item["ITEM_ID"])
all_items = [item["ITEM_ID"] for item in items]

existing_count = session.sql("SELECT COUNT(*) FROM ML_DEMOS.ONLINE_W_STREAMING.RAW_EVENTS").collect()[0][0]

if existing_count > 0:
    print(f"RAW_EVENTS already has {existing_count} rows. Skipping seed.")
else:
    events, ev_sessions = [], []
    while len(events) < 10000:
        while len(ev_sessions) < 15:
            ev_sessions.append(EventSession(user_id=random.choice(user_ids)))
        for s in ev_sessions:
            ev = gen_event(s, users, items_by_cat, all_items)
            if ev: events.append(ev)
        ev_sessions = [s for s in ev_sessions if not s.is_done()]

    now = datetime.now(timezone.utc).replace(tzinfo=None)
    for i, ev in enumerate(events):
        offset = int((i / len(events)) * 48 * 3600)
        ev["EVENT_TS"] = (now - timedelta(seconds=48*3600 - offset)).strftime("%Y-%m-%d %H:%M:%S.%f")

    def _sql_val(v):
        if v is None:
            return "NULL"
        return "'" + str(v).replace("'", "''") + "'"

    batch_size = 500
    for i in range(0, len(events), batch_size):
        batch = events[i:i+batch_size]
        selects = []
        for ev in batch:
            selects.append(
                f"SELECT {_sql_val(ev['EVENT_ID'])}, {_sql_val(ev['SESSION_ID'])}, "
                f"{_sql_val(ev['USER_ID'])}, {_sql_val(ev['ITEM_ID'])}, "
                f"{_sql_val(ev['EVENT_TYPE'])}, {_sql_val(ev['EVENT_TS'])}::TIMESTAMP_NTZ, "
                f"{ev['DWELL_TIME_SEC'] if ev['DWELL_TIME_SEC'] is not None else 'NULL'}, "
                f"{_sql_val(ev['SEARCH_QUERY'])}, {_sql_val(ev['REFERRER_ITEM_ID'])}, "
                f"PARSE_JSON({_sql_val(ev['PROPERTIES'])})"
            )
        sql = (
            "INSERT INTO ML_DEMOS.ONLINE_W_STREAMING.RAW_EVENTS "
            "(EVENT_ID, SESSION_ID, USER_ID, ITEM_ID, EVENT_TYPE, EVENT_TS, "
            "DWELL_TIME_SEC, SEARCH_QUERY, REFERRER_ITEM_ID, PROPERTIES) "
            + " UNION ALL ".join(selects)
        )
        session.sql(sql).collect()

    print(f"Seeded {len(events)} historical events spanning 48 hours.")

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_schema("ML_DEMOS.ONLINE_W_STREAMING")
session.use_warehouse("ML_DEMO_WH")
print(f"Session: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, CreationMode

fs = FeatureStore(
    session=session, database="ML_DEMOS", name="ONLINE_W_STREAMING",
    default_warehouse="ML_DEMO_WH", creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print("Feature Store initialized.")

In [ ]:
user_entity = Entity(name="USER", join_keys=["USER_ID"], desc="Registered clickstream user")
item_entity = Entity(name="ITEM", join_keys=["ITEM_ID"], desc="Product catalog item")
fs.register_entity(user_entity)
fs.register_entity(item_entity)
print("Entities registered.")

In [ ]:
import time
from snowflake.ml.feature_store import online_service

status = fs.get_online_service_status()
if status.status == "RUNNING":
    print("Online service already RUNNING.")
else:
    print("Creating online service...")
    fs.create_online_service("ACCOUNTADMIN", "ACCOUNTADMIN")
    while status.status != "RUNNING":
        time.sleep(30)
        status = fs.get_online_service_status()
        print(f"  Status: {status.status}")

print(f"Ingest URL: {online_service.endpoint_url(status, 'ingest')}")
print(f"Query URL:  {online_service.endpoint_url(status, 'query')}")

In [ ]:
# Feature View 1: USER_PROFILE_FV (batch, online) - training + inference
from snowflake.ml.feature_store import FeatureView, OnlineConfig, OnlineStoreType
from snowflake.snowpark.functions import col, to_timestamp_ntz

user_profile_df = session.table("ML_DEMOS.ONLINE_W_STREAMING.USERS").select(
    col("USER_ID"), col("COUNTRY"), col("USER_SEGMENT"), col("DEVICE_TYPE"),
    col("AGE_GROUP"), to_timestamp_ntz(col("SIGNUP_TS")).alias("SIGNUP_TS")
)

user_profile_fv = FeatureView(
    name="USER_PROFILE_FV", entities=[user_entity], feature_df=user_profile_df,
    timestamp_col="SIGNUP_TS", refresh_freq="1h",
    online_config=OnlineConfig(enable=True, store_type=OnlineStoreType.POSTGRES),
    desc="User demographics - batch, online-enabled for training and inference",
)
registered_user_profile_fv = fs.register_feature_view(user_profile_fv, version="V1")
print(f"Registered: {registered_user_profile_fv.name}/{registered_user_profile_fv.version}")

In [ ]:
# Feature View 2: ITEM_PROFILE_FV (batch, online) - training + inference
item_profile_df = session.table("ML_DEMOS.ONLINE_W_STREAMING.ITEMS").select(
    "ITEM_ID", "CATEGORY", "SUBCATEGORY", "BRAND", "PRICE", "AVG_RATING"
)

item_profile_fv = FeatureView(
    name="ITEM_PROFILE_FV", entities=[item_entity], feature_df=item_profile_df,
    refresh_freq="1h",
    online_config=OnlineConfig(enable=True, store_type=OnlineStoreType.POSTGRES),
    desc="Item catalog attributes - batch, online-enabled for training and inference",
)
registered_item_profile_fv = fs.register_feature_view(item_profile_fv, version="V1")
print(f"Registered: {registered_item_profile_fv.name}/{registered_item_profile_fv.version}")

In [ ]:
# Feature View 3: USER_EVENTS_BATCH_FV (batch, offline) - training only
events_df = session.table("ML_DEMOS.ONLINE_W_STREAMING.RAW_EVENTS").filter(
    "ITEM_ID IS NOT NULL"
).select("USER_ID", "EVENT_ID", "ITEM_ID", "EVENT_TYPE", "EVENT_TS", "DWELL_TIME_SEC", "SESSION_ID")

user_events_batch_fv = FeatureView(
    name="USER_EVENTS_BATCH_FV", entities=[user_entity, item_entity], feature_df=events_df,
    timestamp_col="EVENT_TS", refresh_freq="1 minute",
    desc="User event data for training - batch from RAW_EVENTS, offline only",
)
registered_user_events_batch_fv = fs.register_feature_view(user_events_batch_fv, version="V1")
print(f"Registered: {registered_user_events_batch_fv.name}/{registered_user_events_batch_fv.version}")

In [ ]:
# StreamSource: raw_events
# Name must match the key in ingest payload: {"records": {"raw_events": [...]}}
from snowflake.ml.feature_store import StreamSource
from snowflake.snowpark.types import (
    StructType, StructField, StringType, LongType, TimestampType, TimestampTimeZone,
)

event_stream = StreamSource(
    name="raw_events",
    schema=StructType([
        StructField("EVENT_ID", StringType()),
        StructField("SESSION_ID", StringType()),
        StructField("USER_ID", StringType()),
        StructField("ITEM_ID", StringType()),
        StructField("EVENT_TYPE", StringType()),
        StructField("EVENT_TS", TimestampType(TimestampTimeZone.NTZ)),
        StructField("DWELL_TIME_SEC", LongType()),
        StructField("SEARCH_QUERY", StringType()),
        StructField("REFERRER_ITEM_ID", StringType()),
        StructField("PROPERTIES", StringType()),
    ]),
    desc="Real-time user interaction events - schema matches Ingest API payload",
)
fs.register_stream_source(event_stream)
print("StreamSource registered: raw_events")

In [ ]:
# Feature View 4: USER_EVENTS_STREAM_FV (stream, online) - inference only
from snowflake.ml.feature_store import StreamConfig, Feature
from snowflake.ml.feature_store.spec.enums import FeatureAggregationMethod
import pandas as pd

def transform_events(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["ITEM_ID"].notna()].copy()
    df["IS_VIEW"] = (df["EVENT_TYPE"] == "view").astype(int)
    df["IS_CART"] = (df["EVENT_TYPE"] == "add_to_cart").astype(int)
    df["IS_PURCHASE"] = (df["EVENT_TYPE"] == "purchase").astype(int)
    df["DWELL_TIME_SEC"] = df["DWELL_TIME_SEC"].fillna(0).astype(int)
    return df

backfill_df = session.table("ML_DEMOS.ONLINE_W_STREAMING.RAW_EVENTS").select(
    "EVENT_ID", "SESSION_ID", "USER_ID", "ITEM_ID", "EVENT_TYPE", "EVENT_TS",
    "DWELL_TIME_SEC", "SEARCH_QUERY", "REFERRER_ITEM_ID"
)
stream_cfg = StreamConfig(
    stream_source=event_stream, transformation_fn=transform_events, backfill_df=backfill_df,
)

user_activity_features = [
    Feature.sum("IS_VIEW", "1h").alias("VIEW_COUNT_1H"),
    Feature.sum("IS_VIEW", "24h").alias("VIEW_COUNT_24H"),
    Feature.sum("IS_CART", "1h").alias("CART_COUNT_1H"),
    Feature.sum("IS_PURCHASE", "24h").alias("PURCHASE_COUNT_24H"),
    Feature.sum("DWELL_TIME_SEC", "1h").alias("TOTAL_DWELL_SEC_1H"),
    Feature.approx_count_distinct("ITEM_ID", "1h").alias("UNIQUE_ITEMS_VIEWED_1H"),
]

user_events_stream_fv = FeatureView(
    name="USER_EVENTS_STREAM_FV", entities=[user_entity, item_entity],
    stream_config=stream_cfg, timestamp_col="EVENT_TS",
    refresh_freq="1 minute", feature_granularity="1 minute",
    features=user_activity_features,
    online_config=OnlineConfig(enable=True, store_type=OnlineStoreType.POSTGRES),
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc="Real-time user behavioral aggregations - continuous, online-only, for inference",
)
registered_user_events_stream_fv = fs.register_feature_view(user_events_stream_fv, version="V1")
print(f"Registered: {registered_user_events_stream_fv.name}/{registered_user_events_stream_fv.version}")

In [ ]:
print("\n=== Registered Feature Views ===")
for fv in fs.list_feature_views().collect():
    print(f"  {fv['NAME']} v{fv['VERSION']}")

print("\n=== Endpoints ===")
status = fs.get_online_service_status()
print(f"  Ingest: {online_service.endpoint_url(status, 'ingest')}")
print(f"  Query:  {online_service.endpoint_url(status, 'query')}")

print("\n=== Table Counts ===")
for table in ["USERS", "ITEMS", "RAW_EVENTS"]:
    count = session.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
    print(f"  {table}: {count} rows")

print("\nSetup complete!")